# Prep

## Import stuff

In [1]:
import pandas as pd
pd.set_option('max_colwidth', 100)
import numpy as np
import matplotlib.pyplot as plt
from sklearn import svm, datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import confusion_matrix
import itertools
import scipy.stats as st
from scipy import stats
from sklearn.feature_selection import mutual_info_classif
#import seaborn as sns
#from matplotlib import pyplot as plt
#%matplotlib inline
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 500)
import rpy2
#import pingouin as pg
from itertools import combinations
import openpyxl
from contextlib import redirect_stdout
import random
import math

## Some magical magic to make the R stuff work

In [2]:
# fix from here https://github.com/rpy2/rpy2/issues/882
import os
#os.environ['R_HOME'] = '/Users/zeleninam2/miniconda3/envs/env_hitop_3way_2026/lib/R' # change this to whatever is relevant to you, or you might not need this line at all.

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import rpy2.robjects as ro
from rpy2.robjects.packages import importr
from rpy2.robjects import pandas2ri
import rpy2.ipython.html
rpy2.ipython.html.init_printing()
from rpy2.rinterface_lib.embedded import RRuntimeError

rbase = importr('base')
utils = importr('utils')
lavaan = importr('lavaan')
semtools = importr('semTools')

In [3]:
# importing stuff the right way
try:
    semTools_version = '.'.join([str(ii) for ii in list(ro.r('packageVersion("semTools")')[0])])
except RRuntimeError:
    semTools_version = ''
# There's a bug in the main semtools libarary, this install from my fork of semtools where I've fixed it.
if semTools_version != '0.5.6.933':
    ro.r("""
        library(devtools)
        install_github("shotgunosine/semTools/semTools@aa68e38f88d9dd881617d5ade429bde2b2a74e08")
        """
    )

## Paths

In [4]:
path_to_item_lookup = '../../data/ValSample/Internalizing-Somatoform Items_DW.xlsx'
path_to_cogmood_questions = '../../data/CFA/cogmood_questions.csv'
path_to_valsample = '../../data/ValSample/inters23.xlsx'
path_to_codesheet = '../../dylan_github/yougov_codesheet_alignment.tsv'
path_to_genpop = '../../data/NIMH0007_genpop_num_OUTPUT.csv'
path_to_enriched = '../../data/NIMH0007_mental_health_num_OUTPUT.csv'
path_to_helpfile = '/Users/zeleninam2/Documents/1_projects/hitop/data/CFA/temp/cfa_temp.csv'

## Count how many cpus I have, then decide how many I want to use; define how many iterations for CFA (decrease for debugging)

In [5]:
total_cpus = os.cpu_count()
cpus_to_use = total_cpus-2
global cpus_to_use
print(f"\nGoing to use {cpus_to_use} CPUs for CFA heavy-lifting\n")

num_iter = 1000
global num_iter


Going to use 10 CPUs for CFA heavy-lifting



## SET SEEDS !!!!!!!!!!

In [6]:
#rngkind = "L'Ecuyer-CMRG"
random.seed(12345)

In [7]:
ro.r('RNGkind(kind = "L\'Ecuyer-CMRG")')
ro.r('set.seed(12345)')

<rpy2.rinterface_lib.sexp.NULLType object at 0x157eccad0> [0]

### TEST THE SEEDS!!!!!!!!

In [8]:
for i in range(5):
    print(random.random())
# after kernel restart, this should be 
# 0.41661987254534116
# 0.010169169457068361
# 0.8252065092537432
# 0.2986398551995928
# 0.3684116894884757

0.41661987254534116
0.010169169457068361
0.8252065092537432
0.2986398551995928
0.3684116894884757


In [9]:
ro.r('rnorm(5)')
# after kernel restart, this should be 
# -1.457850350316457	-0.45246126454182867	0.3650586371545244	-1.57091128601566	1.1419085835874878

-1.457850350316457,-0.45246126454182867,0.3650586371545244,-1.57091128601566,1.1419085835874878


### I'M ALSO SETTING THE SAME SEEDS EVERY TIME I RUN THE HELPED CFA FUNCTION, JUST IN CASE!!!!!

# Functions

## Functions for loading and processing data

In [10]:
def make_hitop_list(data):
    # only hitop cols
    hitop_cols = []
    for c in data.columns:
        if 'hitop' in c and 'today' not in c:
            hitop_cols.append(c)   
    return(hitop_cols)

def make_recontact_list(data):
    list_recontact = []
    for c in data.columns:
        if 'recontact' in c:
            list_recontact.append(c)
    return(list_recontact)

def do_checks(dat, remove_checks):
    assert remove_checks in ['no', 'yes', 'grid', 'gridinitialvisitonly']
    dat['passed_checks'] = True
    dat['passed_grid'] = True
    dat['passed_grid_initialonly'] = True
    dat['passed_list'] = True
    dat.loc[dat.check_moderately != 3, 'passed_checks'] = False
    dat.loc[dat.check_notatall != 1, 'passed_checks'] = False
    dat.loc[dat.check_moderately != 3, 'passed_grid'] = False
    dat.loc[dat.check_notatall != 1, 'passed_grid'] = False
    dat.loc[dat.todaycheck_1 != 1, 'passed_checks'] = False
    dat.loc[dat.todaycheck_2 != 1, 'passed_checks'] = False
    dat.loc[dat.todaycheck_1 != 1, 'passed_list'] = False
    dat.loc[dat.todaycheck_2 != 1, 'passed_list'] = False
    # deal with attention checks - recontact
    dat['passed_checks_recontact'] = True
    dat['passed_grid_recontact'] = True
    dat['passed_list_recontact'] = True
    dat.loc[dat.check_moderately_recontact != 3, 'passed_checks_recontact'] = False
    dat.loc[dat.check_notatall_recontact != 1, 'passed_checks_recontact'] = False
    dat.loc[dat.check_moderately_recontact != 3, 'passed_grid_recontact'] = False
    dat.loc[dat.check_notatall_recontact != 1, 'passed_grid_recontact'] = False
    dat.loc[dat.todaycheck_1_recontact != 1, 'passed_checks_recontact'] = False
    dat.loc[dat.todaycheck_2_recontact != 1, 'passed_checks_recontact'] = False
    dat.loc[dat.todaycheck_1_recontact != 1, 'passed_list_recontact'] = False
    dat.loc[dat.todaycheck_2_recontact != 1, 'passed_list_recontact'] = False
    print('checks_initial:')
    print(1 - dat.loc[:, ['passed_checks', 'passed_grid', 'passed_list']].mean())
    how_many_passed_checks = dat['passed_checks'].value_counts()[True]
    how_many_overall = dat.shape[0]
    print('Passed checks: %d out of %d' % (how_many_passed_checks, how_many_overall))
    print('checks_recontact:')
    print(1 - dat.loc[:, ['passed_checks_recontact', 'passed_grid_recontact', 'passed_list_recontact']].mean()) 
    how_many_passed_checks_recontact = dat['passed_checks_recontact'].value_counts()[True]
    print('Passed checks: %d out of %d' % (how_many_passed_checks_recontact, how_many_overall))
    if remove_checks == 'yes':
        print('removing checks')
        dat = dat.loc[dat['passed_checks'] == True]
        dat = dat.loc[dat['passed_checks_recontact'] == True]
        print(dat.shape)
        print('done removing checks')
    elif remove_checks == 'grid':
        print('removing checks')
        dat = dat.loc[dat['passed_grid'] == True]
        dat = dat.loc[dat['passed_grid_recontact'] == True]
        print(dat.shape)
        print('done removing checks') 
    elif remove_checks == 'gridinitialvisitonly':
        print('removing checks')
        dat = dat.loc[dat['passed_grid'] == True]
        print(dat.shape)
        print('done removing checks')        
    return(dat)

def load_item_lookup():
    # this is a table that aligns scale items with their corresponding questions
    htqs = pd.read_excel(path_to_item_lookup, skiprows=2,
                                names=['id', 'scale', 'item', '_0', '_1', '_2', '_3']).drop(['_0', '_1', '_2', '_3'], axis=1)
    # cmqs is a dataframe with items and responses
    # htcm is the same dataframe, but only displaying HiTOP items
    cmqs = pd.read_csv(path_to_cogmood_questions)
    cmqs = cmqs.rename({'Unnamed: 0': 'id'}, axis=1)
    htcm = cmqs.query("measure == 'HiTOP'")
    # this is a lookup table for subscales, items (sentence questions), and hitop ids
    item_lookup = htcm.loc[:, ['id','subscale', 'item']].merge(htqs.loc[:, ['id','item']], how='inner', on='item', suffixes=['_cm', '_ht'])
    item_lookup['htid'] = 'hitop'+ item_lookup.id_ht.astype(str)
    return(item_lookup)
    
def load_valsample():
    # VALIDATION SAMPLE - main dataframe with responses
    dat_validation = pd.read_excel(path_to_valsample)
    '''resp_cols = responses.columns[responses.columns.isin(item_lookup.htid)]
    # scale scores
    scale_scores = []
    for ss, df in item_lookup.groupby('subscale'):
        tmp = responses.loc[:, df.htid].sum(1)
        tmp.name = ss.replace(" ", "_").replace("/", "_").replace("-", "_")
        scale_scores.append(tmp)
    scale_scores = pd.concat(scale_scores, axis=1)
    scale_names = list(scale_scores.columns.values)'''
    #do_checks(dat_validation)    
    # this is the validation data to be used
    hitop_cols_list = make_hitop_list(dat_validation)
    data_validation = dat_validation[hitop_cols_list]
    data_validation = data_validation.assign(whichdata=['validation']*data_validation.shape[0])
    return(data_validation)

def do_codesheet():
    #loads the codesheet file
    # wherever the emane column exists, creates a dict variablename:ename
    code_book = pd.read_csv(path_to_codesheet, sep='\t')
    ename_lut = {vn:en for vn, en in code_book.loc[code_book.ename.notnull(), ['var_name', 'ename']].values}
    for vn, en in code_book.loc[code_book.ename.notnull(), ['var_name', 'ename']].values:
        ename_lut[vn+'_recontact'] = en+'_recontact'
    tmp = code_book.query('ename.notnull()')
    scale_lut = {}
    for ss, df in tmp.loc[tmp.ename.str.contains('hitop') & ~tmp.ename.str.contains('today')].groupby('subscale'):
        scale_name = ss.replace(" ", "_").replace("/", "_").replace("-", "_")
        items = df.ename.values
        scale_lut[scale_name] = items    
     #   scale_name_initial = scale_name+'_initial'
     #   scale_lut[scale_name_initial] = items 
        scale_name_recontact = scale_name+'_recontact'
        items_recontact = []
        for item in items:
            items_recontact.append(item+'_recontact')
        scale_lut[scale_name_recontact] = np.array(items_recontact, dtype=object)
    # rename checks
    ename_lut['FNM_Q8_5'] ='check_moderately'
    ename_lut['FNM_Q22_3'] ='check_notatall'
    ename_lut['FNM_Q42_m_10'] ='todaycheck_1'
    ename_lut['FNM_44_m_28'] ='todaycheck_2'
    ename_lut['FNM_Q8_5_recontact'] ='check_moderately_recontact'
    ename_lut['FNM_Q22_3_recontact'] ='check_notatall_recontact'
    ename_lut['FNM_Q42_m_10_recontact'] ='todaycheck_1_recontact'
    ename_lut['FNM_44_m_28_recontact'] ='todaycheck_2_recontact'
    return ename_lut, scale_lut

def clean_mood_diagnosis(dat):
    # clean mood diagnosis
    dat['mood_dx'] = ''
    dat['n_mood_dx'] = (dat.loc[:, ['FNM_Q25_1', 'FNM_Q25_2', 'FNM_Q25_3', 'FNM_Q25_4', 'FNM_Q25_5', 'FNM_Q25_6']]==1).sum(1)
    dat.loc[dat.FNM_Q25_955 == 1, 'mood_dx'] = 'other'
    dat.loc[dat.FNM_Q25_1 == 1, 'mood_dx'] = 'mdd'
    dat.loc[dat.FNM_Q25_2 == 1, 'mood_dx'] = 'persistent'
    dat.loc[dat.FNM_Q25_3 == 1, 'mood_dx'] = 'premenstrual'
    dat.loc[dat.FNM_Q25_4 == 1, 'mood_dx'] = 'bipolarI'
    dat.loc[dat.FNM_Q25_5 == 1, 'mood_dx'] = 'bipolarII'
    dat.loc[dat.FNM_Q25_6 == 1, 'mood_dx'] = 'cyclothymic'
    dat.loc[dat.n_mood_dx > 1, 'mood_dx'] = 'multiple'
    # clean other mood columns
    dat.loc[dat.mood_years == 999, 'mood_years'] = np.nan
    dat['mood_bothered']=False
    dat.loc[dat.mood_bothered_orig == 1, 'mood_bothered'] = True
    return(dat)

def clean_mood_diagnosis_recontact(dat):
    # clean mood diagnosis - recontact
    dat['mood_dx_recontact'] = ''
    dat['n_mood_dx_recontact'] = (dat.loc[:, ['FNM_Q25_1_recontact', 'FNM_Q25_2_recontact', 'FNM_Q25_3_recontact', 'FNM_Q25_4_recontact', 'FNM_Q25_5_recontact', 'FNM_Q25_6_recontact']]==1).sum(1)
    dat.loc[dat.FNM_Q25_955_recontact == 1, 'mood_dx_recontact'] = 'other'
    dat.loc[dat.FNM_Q25_1_recontact == 1, 'mood_dx_recontact'] = 'mdd'
    dat.loc[dat.FNM_Q25_2_recontact == 1, 'mood_dx_recontact'] = 'persistent'
    dat.loc[dat.FNM_Q25_3_recontact == 1, 'mood_dx_recontact'] = 'premenstrual'
    dat.loc[dat.FNM_Q25_4_recontact == 1, 'mood_dx_recontact'] = 'bipolarI'
    dat.loc[dat.FNM_Q25_5_recontact == 1, 'mood_dx_recontact'] = 'bipolarII'
    dat.loc[dat.FNM_Q25_6_recontact == 1, 'mood_dx_recontact'] = 'cyclothymic'
    dat.loc[dat.n_mood_dx_recontact > 1, 'mood_dx_recontact'] = 'multiple'
    # clean other mood columns - recontact
    # clean other mood columns
    dat.loc[dat.mood_years_recontact == 999, 'mood_years_recontact'] = np.nan
    dat['mood_bothered_recontact']=False
    dat.loc[dat.mood_bothered_recontact_orig == 1, 'mood_bothered_recontact'] = True
    return(dat)

def clean_anxiety_diagnosis(dat):
    # clean anxiety diagnosis
    dat['anxiety_dx'] = ''
    dat['n_anxiety_dx'] = (dat.loc[:, ['FNM_Q30_m_1',
                                       'FNM_Q30_m_2',
                                       'FNM_Q30_m_3', 
                                       'FNM_Q30_m_4', 
                                       'FNM_Q30_m_5',
                                       'FNM_Q30_m_6',
                                       'FNM_Q30_m_7',
                                       'FNM_Q30_m_8',]]==1).sum(1)
    dat.loc[dat.FNM_Q30_m_955 == 1, 'anxiety_dx'] = 'other'
    dat.loc[dat.FNM_Q30_m_1 == 1, 'anxiety_dx'] = 'gad'
    dat.loc[dat.FNM_Q30_m_2 == 1, 'anxiety_dx'] = 'separation'
    dat.loc[dat.FNM_Q30_m_3 == 1, 'anxiety_dx'] = 'agoraphobia'
    dat.loc[dat.FNM_Q30_m_4 == 1, 'anxiety_dx'] = 'phobia'
    dat.loc[dat.FNM_Q30_m_5 == 1, 'anxiety_dx'] = 'social'
    dat.loc[dat.FNM_Q30_m_6 == 1, 'anxiety_dx'] = 'panic_disorder'
    dat.loc[dat.FNM_Q30_m_7 == 1, 'anxiety_dx'] = 'panic_attack'
    dat.loc[dat.FNM_Q30_m_8 == 1, 'anxiety_dx'] = 'mutism'
    dat.loc[dat.n_anxiety_dx > 1, 'anxiety_dx'] = 'multiple'
    # clean other anxiety columns
    dat.loc[dat.anxiety_years == 999, 'anxiety_years'] = np.nan
    dat['anxiety_bothered']=False
    dat.loc[dat.anxiety_bothered_orig == 1, 'anxiety_bothered'] = True
    return(dat)

def clean_anxiety_diagnosis_recontact(dat):
    # clean anxiety diagnosis - recontact
    dat['anxiety_dx_recontact'] = ''
    dat['n_anxiety_dx_recontact'] = (dat.loc[:, ['FNM_Q30_m_1_recontact',
                                       'FNM_Q30_m_2_recontact',
                                       'FNM_Q30_m_3_recontact', 
                                       'FNM_Q30_m_4_recontact', 
                                       'FNM_Q30_m_5_recontact',
                                       'FNM_Q30_m_6_recontact',
                                       'FNM_Q30_m_7_recontact',
                                       'FNM_Q30_m_8_recontact',]]==1).sum(1)
    dat.loc[dat.FNM_Q30_m_955_recontact == 1, 'anxiety_dx_recontact'] = 'other'
    dat.loc[dat.FNM_Q30_m_1_recontact == 1, 'anxiety_dx_recontact'] = 'gad'
    dat.loc[dat.FNM_Q30_m_2_recontact == 1, 'anxiety_dx_recontact'] = 'separation'
    dat.loc[dat.FNM_Q30_m_3_recontact == 1, 'anxiety_dx_recontact'] = 'agoraphobia'
    dat.loc[dat.FNM_Q30_m_4_recontact == 1, 'anxiety_dx_recontact'] = 'phobia'
    dat.loc[dat.FNM_Q30_m_5_recontact == 1, 'anxiety_dx_recontact'] = 'social'
    dat.loc[dat.FNM_Q30_m_6_recontact == 1, 'anxiety_dx_recontact'] = 'panic_disorder'
    dat.loc[dat.FNM_Q30_m_7_recontact == 1, 'anxiety_dx_recontact'] = 'panic_attack'
    dat.loc[dat.FNM_Q30_m_8_recontact == 1, 'anxiety_dx_recontact'] = 'mutism'
    
    dat.loc[dat.n_anxiety_dx_recontact > 1, 'anxiety_dx_recontact'] = 'multiple'
    # clean other anxiety columns - recontact
    dat.loc[dat.anxiety_years_recontact == 999, 'anxiety_years_recontact'] = np.nan
    dat['anxiety_bothered_recontact']=False
    dat.loc[dat.anxiety_bothered_recontact_orig == 1, 'anxiety_bothered_recontact'] = True
    return(dat)

def clean_attnt_dx(dat):
    # clean attention diagnosis
    dat['attention_dx'] = ''
    dat['n_attention_dx'] = (dat.loc[:, ['FNM_Q35_m_1',
                                       'FNM_Q35_m_2',
                                      ]]==1).sum(1)
    dat.loc[dat.FNM_Q35_m_3 == 1, 'attention_dx'] = 'other'
    dat.loc[dat.FNM_Q35_m_1 == 1, 'attention_dx'] = 'adhd'
    dat.loc[dat.FNM_Q35_m_2 == 1, 'attention_dx'] = 'adhd'
    
    # clean other attention columns
    dat.loc[dat.attention_years == 999, 'attention_years'] = np.nan
    dat['attention_bothered']=False
    dat.loc[dat.attention_bothered_orig == 1, 'attention_bothered'] = True
    return(dat)

def clean_attnt_dx_recontact(dat):
    # clean attention diagnosis - recontact
    dat['attention_dx_recontact'] = ''
    dat['n_attention_dx_recontact'] = (dat.loc[:, ['FNM_Q35_m_1_recontact',
                                       'FNM_Q35_m_2_recontact',
                                      ]]==1).sum(1)
    dat.loc[dat.FNM_Q35_m_3_recontact == 1, 'attention_dx_recontact'] = 'other'
    dat.loc[dat.FNM_Q35_m_1_recontact == 1, 'attention_dx_recontact'] = 'adhd'
    dat.loc[dat.FNM_Q35_m_2_recontact == 1, 'attention_dx_recontact'] = 'adhd'
    
    # clean other attention columns - recontact
    dat.loc[dat.attention_years_recontact == 999, 'attention_years_recontact'] = np.nan
    dat['attention_bothered_recontact']=False
    dat.loc[dat.attention_bothered_recontact_orig == 1, 'attention_bothered_recontact'] = True
    return(dat)

def define_attnt_checks(dat):
    # deal with attention checks 
    dat['passed_checks'] = True
    dat['passed_grid'] = True
    dat['passed_list'] = True
    dat.loc[dat.check_moderately != 3, 'passed_checks'] = False
    dat.loc[dat.check_notatall != 1, 'passed_checks'] = False
    dat.loc[dat.check_moderately != 3, 'passed_grid'] = False
    dat.loc[dat.check_notatall != 1, 'passed_grid'] = False
    dat.loc[dat.todaycheck_1 != 1, 'passed_checks'] = False
    dat.loc[dat.todaycheck_2 != 1, 'passed_checks'] = False
    dat.loc[dat.todaycheck_1 != 1, 'passed_list'] = False
    dat.loc[dat.todaycheck_2 != 1, 'passed_list'] = False 
    # deal with attention checks - recontact
    dat['passed_checks_recontact'] = True
    dat['passed_grid_recontact'] = True
    dat['passed_list_recontact'] = True
    dat.loc[dat.check_moderately_recontact != 3, 'passed_checks_recontact'] = False
    dat.loc[dat.check_notatall_recontact != 1, 'passed_checks_recontact'] = False
    dat.loc[dat.check_moderately_recontact != 3, 'passed_grid_recontact'] = False
    dat.loc[dat.check_notatall_recontact != 1, 'passed_grid_recontact'] = False
    dat.loc[dat.todaycheck_1_recontact != 1, 'passed_checks_recontact'] = False
    dat.loc[dat.todaycheck_2_recontact != 1, 'passed_checks_recontact'] = False
    dat.loc[dat.todaycheck_1_recontact != 1, 'passed_list_recontact'] = False
    dat.loc[dat.todaycheck_2_recontact != 1, 'passed_list_recontact'] = False
    # how many passed checks?
    print('\nThis is how many ppl passed attention checks:')
    print('Calculated as 1 - dat.loc[:, [\'passed_checks\', \'passed_grid\', \'passed_list\']].mean()')
    print(1 - dat.loc[:, ['passed_checks', 'passed_grid', 'passed_list']].mean())
    print('Same for recontact:')
    print(1 - dat.loc[:, ['passed_checks_recontact', 'passed_grid_recontact', 'passed_list_recontact']].mean())
    return(dat)
    
def load_data(dataname, doing_checks, doing_remove_checks):
    print(doing_remove_checks)
    assert(dataname in ['genpop','enriched'])
    
    #codesheet file
    ename_lut, scale_lut = do_codesheet()
    
    # done working with the codesheet file, now opening the actual data
    if dataname == 'genpop':
        datapath = path_to_genpop
    elif dataname == 'enriched':
        datapath = path_to_enriched
    dat = pd.read_csv(datapath, dtype={'caseid':str}, engine='python')
    if dataname == 'genpop':
        # ONLY GENPOP drop the .0 that pandas appends for some reason - only for genpop
        dat['caseid'] = dat.caseid.str[:-2]
    
    #renaming columns
    dat = dat.rename(ename_lut, axis=1)
    dat = dat.rename(columns={
        'mood_bothered': 'mood_bothered_orig',
        'mood_bothered_recontact': 'mood_bothered_recontact_orig',
        'anxiety_bothered': 'anxiety_bothered_orig',
        'anxiety_bothered_recontact': 'anxiety_bothered_recontact_orig',
        'attention_bothered': 'attention_bothered_orig',
        'attention_bothered_recontact': 'attention_bothered_recontact_orig'}) 
    
    # clean diagnoses
    dat = clean_mood_diagnosis(dat)
    dat = clean_mood_diagnosis_recontact(dat)
    dat = clean_anxiety_diagnosis(dat)
    dat = clean_anxiety_diagnosis_recontact(dat)
    dat = clean_attnt_dx(dat)
    dat = clean_attnt_dx_recontact(dat)
    
    if doing_checks:
        dat = do_checks(dat = dat, remove_checks = doing_remove_checks) 

    minus_1_cols = []
    m1_stems = ['inattention', 'hyperactivity', 'impulsivity', 'sct', 'gad', 'phq', 'hitop',
           'inattention_recontact', 'hyperactivity_recontact', 'impulsivity_recontact', 'sct_recontact', 'gad_recontact', 'phq_recontact', 'hitop_recontact']
    for ms in m1_stems:
        #print(ms)
        if "recontact" not in ms:
            cols = list(dat.columns[dat.columns.str.contains(ms) & ~dat.columns.str.contains('today') & ~dat.columns.str.contains('sum') & ~dat.columns.str.contains('recontact')].values)
        else:
            cols = list(dat.columns[dat.columns.str.contains(ms[:-10]) & ~dat.columns.str.contains('today') & ~dat.columns.str.contains('sum') & dat.columns.str.contains('recontact')].values)
        #print(cols)
        minus_1_cols.extend(cols)

    # why are we doing this
    # subtract 1 from responses
    dat.loc[:, minus_1_cols] -= 1

    # summing up the values
    sum_cols = []
    for ms in m1_stems:
        if 'hitop' not in ms:
            if "recontact" not in ms:
                cols = list(dat.columns[dat.columns.str.contains(ms) & ~dat.columns.str.contains('today') & ~dat.columns.str.contains('sum') & ~dat.columns.str.contains('recontact')].values)
                dat[f'{ms}_sum'] = dat.loc[:, cols].sum(1)
                sum_cols.append(ms + '_sum')
            else:
                cols = list(dat.columns[dat.columns.str.contains(ms[:-10]) & ~dat.columns.str.contains('today') & ~dat.columns.str.contains('sum') & dat.columns.str.contains('recontact')].values)
                dat[f'{ms[:-10]}_sum_recontact'] = dat.loc[:, cols].sum(1)
                sum_cols.append(ms[:-10] + '_sum_recontact')    
    hitop_sums = []
    for scale_name, items in scale_lut.items():
        dat[scale_name] = dat.loc[:, items].sum(1) # adding a dat[scale_name] with a sum of all values ("items)"
        hitop_sums.append(scale_name)   

    sum = 0
    for scale_name, items in scale_lut.items():
        if "well_being" not in scale_name:
            sum += len(items)

    # !!!!!!!!!! double-check this
    dat['hitop_sum'] = dat.loc[:, hitop_sums[::2][:-1]].sum(1)  # [:-1] because we don't include the well-being scale
    dat['hitop_sum_recontact'] = dat.loc[:, hitop_sums[1::2][:-1]].sum(1)
    dat['baars_sum'] = dat.inattention_sum + dat.hyperactivity_sum + dat.impulsivity_sum
    dat['baars_sum_recontact'] = dat.inattention_sum_recontact + dat.hyperactivity_sum_recontact + dat.impulsivity_sum_recontact
    dat['moodanxiety_bothered'] = dat.mood_bothered | dat.anxiety_bothered
    dat['moodanxiety_bothered_recontact'] = dat.mood_bothered_recontact | dat.anxiety_bothered_recontact

    my_columns = []
    for item in ['hitop_sum', 'baars_sum', 'phq_sum', 'gad_sum', # all sums
                 'mood_bothered','anxiety_bothered', 'attention_bothered', 'moodanxiety_bothered', # bothered
                 'inattention_sum', 'hyperactivity_sum', 'impulsivity_sum', 'sct_sum']: # each subscale of baars
        my_columns.append(item)
        my_columns.append(item+'_recontact')
    my_columns.extend(hitop_sums) #each subscale of hitop

        # rename columns with phq and gad sums for cnsistency in naming
    dat = dat.rename(columns={"gad_recontact_sum": "gad_sum_recontact", "phq_recontact_sum": "phq_sum_recontact"})

    # rename baars subscales
    dat = dat.rename(columns={"inattention_sum": "baars_inattention_sum", 
                          "inattention_sum_recontact": "baars_inattention_sum_recontact",
                          "hyperactivity_sum": "baars_hyperactivity_sum", 
                          "hyperactivity_sum_recontact": "baars_hyperactivity_sum_recontact",
                          "impulsivity_sum": "baars_impulsivity_sum", 
                          "impulsivity_sum_recontact": "baars_impulsivity_sum_recontact",
                          "sct_sum": "baars_sct_sum",
                          "sct_sum_recontact": "baars_sct_sum_recontact"})

    # rename hitops subscales
    rename_dict = {}
    for hitop_item in hitop_sums:
        rename_dict[hitop_item] = 'hitop_' + hitop_item
    dat = dat.rename(columns=rename_dict)
    
    hitop_cols_list = make_hitop_list(dat)
    dat = dat.assign(whichdata=[dataname]*dat.shape[0])
    data = dat[hitop_cols_list]
    data = data.assign(whichdata=[dataname]*data.shape[0])
    recontact_cols_list = make_recontact_list(data)
    data_norecontact = data.drop(columns=recontact_cols_list)
    return dat, data, data_norecontact   

def prep_data_recontactanalysis(dataframe_withrecontact):
    df = dataframe_withrecontact
    df_helper = df.copy(deep=True)
    df_rows = df.shape[0]
    df = df.assign(whichvisit=['initialvisit']*df_rows)
    df_helper = df_helper.assign(whichvisit=['recontactvisit']*df_rows)
    # in df, drop all "recontact" columns
    # in df_helper, drop all initial visit columns, then rename the recontact columns
    cols_recontact = []
    cols_original = []
    for c in df.columns:
        if "recontact" in c:
            cols_recontact.append(c)
            cols_original.append(c[:-10])  
    df = df.drop(columns=cols_recontact)
    df_helper = df_helper.drop(columns=cols_original)
    cols_dict = {}
    for cname in cols_recontact:
        cols_dict[cname] = cname[:-10]
    # rename cols in df_helper
    df_helper.rename(columns=cols_dict, inplace=True)    
    mydata_withrecontact = pd.concat([df, df_helper])
    return(mydata_withrecontact)

## CFA helper functions

In [11]:
def print_summary(obj):
    return rprint(summary(obj))

def print_short_summary(invariance_output):
    ro.r("myoutput <- capture.output(summary(invariance_output))") 
    ro.r("stringid <- grep(\"chisq\", myoutput)")
    ro.r("print(myoutput[stringid-1])")
    ro.r("print(myoutput[stringid])")
    return (0)

def print_problematic_items(invariance_output):
    ro.r("myoutput <- capture.output(summary(invariance_output))") 
    print('Problematic items:')
    ro.r("stringidsig <- grep(\"may differ between Groups\", myoutput)")
    ro.r("print(myoutput[stringidsig])")   
    return (0)
    
def extract_p(invariance_output):
    ro.globalenv['invariance_output'] = invariance_output
    ro.r("myoutput <- capture.output(summary(invariance_output))") 
    ro.r("stringid <- grep(\"chisq\", myoutput)")
    ro.r('myrline <- myoutput[stringid]')
    mypline = ro.r('myrline')[0]
    my_p = mypline.split()[-1]
    return(my_p)

def check_secondary_criteria(fit_config):
    Criteria_passed = False
    ro.r("myoutputconfig <- capture.output(summary(fit_config, fit.measures=TRUE))")
    ro.r("stringid_cfi <- grep(\"Robust.*CFI\", myoutputconfig)")
    ro.r("cfi_line <- myoutputconfig[stringid_cfi]")
    cfiline = ro.r('cfi_line')[0]    
    my_cfi = cfiline.split()[-1] 
    my_cfi = float(my_cfi)
    ro.r("stringid_tli <- grep(\"Robust.*TLI\", myoutputconfig)")
    ro.r("tli_line <- myoutputconfig[stringid_tli]")
    tliline = ro.r('tli_line')[0]    
    my_tli = tliline.split()[-1] 
    my_tli = float(my_tli)
    ro.r("stringid_rmsea <- grep(\"Robust.*RMSEA\", myoutputconfig)")
    ro.r("rmsea_line <- myoutputconfig[stringid_rmsea]")
    myrmsealine = ro.r('rmsea_line')[0]
    my_rmsea = myrmsealine.split('\n')[0].split()[-1]
    my_rmsea = float(my_rmsea)   
    if (my_cfi > 0.95) and (my_tli > 0.95) and (my_rmsea < 0.06):
        Criteria_passed = True
    return (Criteria_passed, my_cfi, my_tli, my_rmsea)   

def check_hitop_ids(items_to_check, my_item_lookup):
    # checks what item ids mean what
    for i in items_to_check:
        a = repr(my_item_lookup.loc[my_item_lookup['htid'] == 'hitop'+i]['htid']).split(' ', 1)[1]
        aa = str(a).split('\n')[0]
        b = repr(my_item_lookup.loc[my_item_lookup['htid'] == 'hitop'+i]['item']).split(' ', 1)[1]
        bb = str(b).split('\n')[0]
        print(aa+bb)
    print('\n')

def cfa_helper_func(scalename, list_of_items, do_metric, do_scalar, do_strict, mydata_python, mydata_temp_path):
    # Helper CFA funct
    # Inputs:
    # --> scalename - string of scale name: 'appetite_loss'
    # --> list_of_items - list of items to test, not necessarily the full list of items for the scale: ['hitop280', 'hitop283', 'hitop109']
    # --> do_metric, do_scalar, do_strict: True/False
    # --> mydata_python - python df with data (this function will feed it to R)
    # --> mydata_temp_path - path to do an intermediate step of saving from python, then reading into R

    # SETTING THE SEED HERE, JUST TO BE SURE IT IS SET TO THE SAME THING EVERY TIME I RUN THIS HELPER FUNCTION
    random.seed(12345)
    ro.r('RNGkind(kind = "L\'Ecuyer-CMRG")')
    ro.r('set.seed(12345)')

    rprint = ro.r('print')
    summary = ro.r('summary')

    # make a R-readable string out of scalename and desired list of items
    # (this will sometimes match the full set - and it's okay - but sometimes won't, when we iterate through all options looking for inv subsets)
    myrstring = scalename + "=~"
    for x in list_of_items: # iterate through desired items and add them to my formula
        myrstring += x + ' + '
    myrstring = myrstring[:-3] # drop the last " + "
    #print(f'Testing formula: {myrstring}')      
    ro.globalenv['testformula_r'] = myrstring # send it to the R world

    # transfer data from python into R
    # I save the python df as csv and then load it to R (there must be a more elegant way to do this, but this one works...)
    mydata_python.to_csv(mydata_temp_path)
    r_data_command = f'rdata <- read.csv(\'{mydata_temp_path}\', header=TRUE)'
    ro.r(r_data_command)

    # telling R which column to test
    group = 'whichdata'
    ro.globalenv['group'] = group

    flag_passed_metric = False # setting flag for metric because this level is the most important one --> we return this
    
    # Always do CONFIGURAL:
    flag_passed_config = False # flag for configural
    fit_config = ro.r('cfa(testformula_r, data = rdata, group = group, estimator = "WLSMV")')
    ro.globalenv['fit_config'] = fit_config        
    out_config = semtools.permuteMeasEq(nPermute=num_iter,
                                        con=fit_config, # In the case of testing configural invariance when modelType = "mgcfa", con is the configural model (implicitly, the unconstrained model is the saturated model, so use the defaults uncon = NULL and param = NULL).
                                        parallelType="multicore", ncpus=cpus_to_use)
    config_p = extract_p(out_config)
    if float(config_p) >= 0.05:
        print('CONFIG INVARIANT chisq p = ' + str(config_p))
        flag_passed_config = True
    else:
        (flag_passed_config, my_cfi, my_tli, my_rmsea) = check_secondary_criteria(fit_config)
        if flag_passed_config:
            print('CONFIG chisq p = ' + str(config_p))
            print('PASSED SECONDARY CRITERIA WITH CFI = ' + str(my_cfi) + ' TLI = ' + str(my_tli) + ' RMSEA = ' + str(my_rmsea))

    if flag_passed_config: # only do metric if configural is passed
        if do_metric:
            # ====== METRIC =====
            fit_metric = ro.r('cfa(testformula_r, data = rdata, group = group, estimator = "WLSMV", group.equal="loadings")')
            ro.globalenv['fit_metric'] = fit_metric
            out_metric = semtools.permuteMeasEq(nPermute=num_iter, 
                                                        uncon=fit_config, 
                                                        con=fit_metric, 
                                                        param="loadings",
                                                        parallelType="multicore", ncpus=cpus_to_use)
            metric_p = extract_p(out_metric)
            if float(metric_p) >= 0.05:
                print('METRIC INVARIANT chisq p = ' + str(metric_p))
                flag_passed_metric = True

                if do_scalar: # only do scalar if metric is passed
                    # ====== SCALAR =====
                    fit_scalar = ro.r('cfa(testformula_r, data = rdata, group = group, estimator = "WLSMV", group.equal=c("loadings", "intercepts"))')
                    ro.globalenv['fit_scalar'] = fit_scalar
                    out_scalar = semtools.permuteMeasEq(nPermute=num_iter, 
                                                            uncon=fit_metric, 
                                                            con=fit_scalar, 
                                                            param=ro.StrVector(["loadings","intercepts"]),
                                                            parallelType="multicore", 
                                                            ncpus=cpus_to_use)
                    scalar_p = extract_p(out_scalar)
                    if float(scalar_p) >= 0.05:
                        print('SCALAR INVARIANT chisq p = ' + str(scalar_p)) 
                        
                        if do_strict: # only do strict if scalar is passed
                            # ====== STRICT =====
                            ro.globalenv['out_scalar'] = out_scalar
                            fit_strict = ro.r('cfa(testformula_r, data = rdata, group = group, estimator = "WLSMV", group.equal=c("loadings", "intercepts", "residuals"))')
                            ro.globalenv['fit_strict'] = fit_strict
                            out_strict = semtools.permuteMeasEq(nPermute=num_iter, 
                                                                        uncon=fit_scalar, 
                                                                        con=fit_strict, 
                                                                        param=ro.StrVector(["loadings","intercepts", "residuals"]),
                                                                        parallelType="multicore", 
                                                                        ncpus=cpus_to_use)
                            strict_p = extract_p(out_strict)
                            if float(strict_p) >= 0.05:
                                print('STRICT INVARIANT chisq p = ' + str(strict_p))
                            else:
                                # significant:
                                print('STRICT INVARIANT NOT PASSED, chisq p = ' + str(strict_p))
                        else: # not do_strict
                            print('STRICT N/A')
                            strict_p = 'NA'
                    else: # scalar not passed
                        print('SCALAR INVARIANT NOT PASSED, chisq p = ' + str(scalar_p))
                        print('STRICT N/A')
                        strict_p = 'NA' 
                else: # not do_scalar
                    print('SCALAR N/A')
                    scalar_p = 'NA' 
                    print('STRICT N/A')
                    strict_p = 'NA'                     
            else: # metric not passed
                print('METRIC INVARIANT NOT PASSED, chisq p = ' + str(metric_p)) 
                print('SCALAR N/A')
                print('STRICT N/A')                    
                scalar_p = 'NA'
                strict_p = 'NA'       
        else: # not do_metric
            print('METRIC N/A')
            print('SCALAR N/A')
            print('STRICT N/A')
            metric_p = 'NA'
            scalar_p = 'NA'
            strict_p = 'NA'            
    else: # config not passed
        print('CONFIG INVARIANT NOT PASSED, chisq p = ' + str(config_p))
        print('METRIC N/A')
        print('SCALAR N/A')
        print('STRICT N/A')
        metric_p = 'NA'
        scalar_p = 'NA'
        strict_p = 'NA'
    
    return(flag_passed_metric, config_p, metric_p, scalar_p, strict_p)

## CFA wrapper functions

In [12]:
def do_three_way_cfa_ablations(whichscale, whichcfa, howmanyitems):
    
    # checks all combinations of X items for Y scale,
    # reports if a 3-way invariant combination was found
    
    print(f'FOR SCALE {whichscale.upper()}:')    
    print(f'RUN ALL COMBINATIONS OF {howmanyitems} ITEMS and checks if a combination is 3-way invariant.')
    
    flag_found_inv_in_all_three = False
    successful_combinations = {}

    # these are the original scales
    mdl_strs_new = {'anhedonic_depression': 'anhedonic_depression =~hitop39 + hitop77 + hitop84 + hitop92 + hitop93 + hitop123 + hitop157 + hitop182 + hitop230 + hitop246',
         'anxious_worry': 'anxious_worry =~hitop20 + hitop34 + hitop89 + hitop203 + hitop248 + hitop265',
         'appetite_gain': 'appetite_gain =~hitop120 + hitop141 + hitop243 + hitop275',
         'appetite_loss': 'appetite_loss =~hitop280 + hitop283 + hitop109',
         'cognitive_problems': 'cognitive_problems =~hitop67 + hitop189 + hitop142',
         'hyposomnia': 'hyposomnia =~hitop99 + hitop181 + hitop5 + hitop66 + hitop231',
         'insomnia': 'insomnia =~hitop160 + hitop254 + hitop261 + hitop268',
         'panic': 'panic =~hitop15 + hitop104 + hitop126 + hitop211 + hitop215 + hitop257',
         'separation_insecurity': 'separation_insecurity =~hitop40 + hitop50 + hitop69 + hitop81 + hitop113 + hitop136 + hitop151 + hitop197',
         'shame_guilt': 'shame_guilt =~hitop72 + hitop140 + hitop143 + hitop220',
         'situational_phobia': 'situational_phobia =~hitop16 + hitop165 + hitop225 + hitop247 + hitop278',
         'social_anxiety': 'social_anxiety =~hitop1 + hitop17 + hitop114 + hitop117 + hitop124 + hitop129 + hitop204 + hitop222 + hitop236 + hitop258',
         'well_being': 'well_being =~hitop9 + hitop23 + hitop54 + hitop106 + hitop149 + hitop200 + hitop244 + hitop245 + hitop250 + hitop281'}
    
    for cln_ss, mdl in mdl_strs_new.items(): 
        if cln_ss == whichscale:  # take the scale we are interested in 
            if whichcfa == 'metric':
                doing_metric = True
                doing_scalar = False
                doing_strict = False
            elif whichcfa == 'scalar':
                doing_metric = True
                doing_scalar = True
                doing_strict = False   
            elif whichcfa == 'strict':
                doing_metric = True
                doing_scalar = True
                doing_strict = True                  
            print("\n")
            print(f"Running {cln_ss.upper()}")
            print(f"Items: {mdl}")
            
            # loop to do ablations
            temp_items = mdl_strs_new[cln_ss]
            temp_items_items = temp_items.split("=~",1)[1]
            temp_items_items_items = temp_items_items.split(" + ")
            list_of_items = temp_items_items_items

            count_success = 0
            count_comb = 1

            for com in combinations(list_of_items, howmanyitems):

                # number of possible combinations of x items of of n possible items
                num_combinations = math.comb(len(list_of_items), howmanyitems)
                print(f'\n+++ TESTING {count_comb}th COMBINATION out of {num_combinations} possible combinations of {howmanyitems} items +++')
                print(f'Items to test: {com}')

                # set flags for metric invariance genpop and enriched to FALSE, for this combination of items
                flag_metric_val_gp = False
                flag_metric_val_en = False
                flag_metric_gp_en = False 

                # +++ VALIDATION VS GENPOP +++
                print('\n -----> VALID VS GENPOP <----- ')
                flag_metric_val_gp, pconfig_val_gp, pmetric_val_gp, pscalar_val_gp, pstrict_val_gp = cfa_helper_func(\
                    scalename = cln_ss,\
                    list_of_items = com,\
                    do_metric = doing_metric, \
                    do_scalar = doing_scalar, \
                    do_strict = doing_strict, \
                    mydata_python = mydata_val_genpop_norecontact, \
                    mydata_temp_path = path_to_helpfile)
            
                # +++ VALIDATION VS ENRICHED +++ 
                print('\n -----> VALID VS ENRICHED <----- ')
                flag_metric_val_en, pconfig_val_en, pmetric_val_en, pscalar_val_en, pstrict_val_en = cfa_helper_func(\
                    scalename = cln_ss,\
                    list_of_items = com,\
                    do_metric = doing_metric, \
                    do_scalar = doing_scalar, \
                    do_strict = doing_strict, \
                    mydata_python = mydata_val_enriched_norecontact, \
                    mydata_temp_path = path_to_helpfile)

                # +++ GENPOP VS ENRICHED +++ 
                print('\n -----> GENPOP VS ENRICHED <----- ')
                flag_metric_gp_en, pconfig_gp_en, pmetric_gp_en, pscalar_gp_en, pstrict_gp_en = cfa_helper_func(\
                    scalename = cln_ss,\
                    list_of_items = com,\
                    do_metric = doing_metric, \
                    do_scalar = doing_scalar, \
                    do_strict = doing_strict, \
                    mydata_python = data_genpop_enriched_norecontact, \
                    mydata_temp_path = path_to_helpfile)

                if all([flag_metric_val_gp, flag_metric_val_en, flag_metric_gp_en]):
                    print(f"Combination {count_comb} passes 3-way invariance!")
                    flag_found_inv_in_all_three = True
                    count_success += 1
                    successful_combinations[com]={'val_gp':[pconfig_val_gp, pmetric_val_gp, pscalar_val_gp, pstrict_val_gp],\
                                                  'val_en':[pconfig_val_en, pmetric_val_en, pscalar_val_en, pstrict_val_en],\
                                                  'gp_en':[pconfig_gp_en, pmetric_gp_en, pscalar_gp_en, pstrict_gp_en]}
                count_comb+=1
                        
            print("\nCFA DONE")
            if flag_found_inv_in_all_three:
                print(f'\n\nFound {count_success} combination(s) of {howmanyitems} that is 3-way invariant.')
                print(successful_combinations)
            
            else:
                print(f'\nCould not find a combination of {howmanyitems} for scale {whichscale} that is 3-way invariant =(')
                print(f'Val–GP: {flag_metric_val_gp}')
                print(f'Val–EN: {flag_metric_val_en}')
                print(f'GP–EN:  {flag_metric_gp_en}\n')
                
    return(successful_combinations)

In [13]:
def run_specific_cfa(whichscale, item_list, whichcfa):
    # checks 3-way CFA for a specific formula
    print(f'\n\nFOR SCALE {whichscale.upper()}:')    
    print(f'\nRUN ALL 3-way CFA for items: {item_list}.')
    
    # down to which scale to test
    if whichcfa == 'metric':
        doing_metric = True
        doing_scalar = False
        doing_strict = False
    elif whichcfa == 'scalar':
        doing_metric = True
        doing_scalar = True
        doing_strict = False   
    elif whichcfa == 'strict':
        doing_metric = True
        doing_scalar = True
        doing_strict = True  

    # set flags for metric invariance to FALSE, for this combination of items
    flag_metric_val_gp = False
    flag_metric_val_en = False
    flag_metric_gp_en = False 

    # +++ VALIDATION VS GENPOP +++
    print('\n -----> VALID VS GENPOP <----- ')
    flag_metric_val_gp, pconfig_val_gp, pmetric_val_gp, pscalar_val_gp, pstrict_val_gp = cfa_helper_func(\
                    scalename = whichscale,\
                    list_of_items = item_list,\
                    do_metric = doing_metric, \
                    do_scalar = doing_scalar, \
                    do_strict = doing_strict, \
                    mydata_python = mydata_val_genpop_norecontact, \
                    mydata_temp_path = path_to_helpfile)
            
    # +++ VALIDATION VS ENRICHED +++ 
    print('\n -----> VALID VS ENRICHED <----- ')
    flag_metric_val_en, pconfig_val_en, pmetric_val_en, pscalar_val_en, pstrict_val_en = cfa_helper_func(\
                    scalename = whichscale,\
                    list_of_items = item_list,\
                    do_metric = doing_metric, \
                    do_scalar = doing_scalar, \
                    do_strict = doing_strict, \
                    mydata_python = mydata_val_enriched_norecontact, \
                    mydata_temp_path = path_to_helpfile)

    # +++ GENPOP VS ENRICHED +++ 
    print('\n -----> GENPOP VS ENRICHED <----- ')
    flag_metric_gp_en, pconfig_gp_en, pmetric_gp_en, pscalar_gp_en, pstrict_gp_en = cfa_helper_func(\
                    scalename = whichscale,\
                    list_of_items = item_list,\
                    do_metric = doing_metric, \
                    do_scalar = doing_scalar, \
                    do_strict = doing_strict, \
                    mydata_python = data_genpop_enriched_norecontact, \
                    mydata_temp_path = path_to_helpfile)

    if all([flag_metric_val_gp, flag_metric_val_en, flag_metric_gp_en]):
        flag_found_inv_in_all_three = True
        print("\n!!! PASSES 3-WAY INVARIANCE")
                   
    print("DONE")

    return(0)

# Run Main Code

## Load all data

In [14]:
# Load item_lookup
item_lookup = load_item_lookup()

# Load validation data 
data_val = load_valsample()

# Load genpop data
data_genpop_full, data_genpop, data_genpop_norecontact = load_data('genpop', doing_checks = True, doing_remove_checks='gridinitialvisitonly')

# Load enriched data
data_enriched_full, data_enriched, data_enriched_norecontact = load_data('enriched', doing_checks = True, doing_remove_checks='gridinitialvisitonly')

# Concatinate data
mydata_val_genpop = pd.concat([data_val, data_genpop])
mydata_val_genpop_norecontact = pd.concat([data_val, data_genpop_norecontact])

mydata_val_enriched = pd.concat([data_val, data_enriched])
mydata_val_enriched_norecontact = pd.concat([data_val, data_enriched_norecontact])

data_genpop_enriched_norecontact = pd.concat([data_genpop_norecontact, data_enriched_norecontact])
data_genpop_enriched_full = pd.concat([data_genpop_full, data_enriched_full])

# recontact
data_genpop_recontactanalysis = prep_data_recontactanalysis(data_genpop)
data_enriched_recontactanalysis = prep_data_recontactanalysis(data_enriched)

/Users/zeleninam2/miniconda3/envs/env_semtools_fix/lib/python3.11/site-packages/openpyxl/worksheet/_read_only.py:85: UserWarning: Unknown extension is not supported and will be removed
  for idx, row in parser.parse():


gridinitialvisitonly
checks_initial:
passed_checks    0.462
passed_grid      0.006
passed_list      0.458
dtype: float64
Passed checks: 269 out of 500
checks_recontact:
passed_checks_recontact    0.548
passed_grid_recontact      0.200
passed_list_recontact      0.548
dtype: float64
Passed checks: 226 out of 500
removing checks
(497, 753)
done removing checks
gridinitialvisitonly
checks_initial:
passed_checks    0.293548
passed_grid      0.016129
passed_list      0.290323
dtype: float64
Passed checks: 219 out of 310
checks_recontact:
passed_checks_recontact    0.412903
passed_grid_recontact      0.174194
passed_list_recontact      0.412903
dtype: float64
Passed checks: 182 out of 310
removing checks
(305, 753)
done removing checks


In [58]:
# check nans
for d in [data_val, data_genpop, data_genpop_norecontact,
         data_enriched, data_enriched_norecontact,
         mydata_val_genpop_norecontact,
         mydata_val_enriched_norecontact,
         data_genpop_enriched_norecontact, 
         data_genpop_recontactanalysis, data_enriched_recontactanalysis]:
    print('checking for nans')
    print(d.isnull().values.any())

print('\n')

checking for nans
False
checking for nans
False
checking for nans
False
checking for nans
False
checking for nans
False
checking for nans
True
checking for nans
True
checking for nans
False
checking for nans
False
checking for nans
False




In [43]:
data_genpop_full.shape

(497, 800)

In [44]:
data_genpop.shape

(497, 197)

In [45]:
data_genpop_norecontact.shape

(497, 99)

In [52]:
data_genpop_enriched_full.shape

(802, 800)

In [46]:
data_genpop_enriched_norecontact.head()

,hitop157,hitop81,hitop34,hitop54,hitop243,hitop182,hitop69,hitop89,hitop50,hitop129,hitop265,hitop124,hitop231,hitop93,hitop67,hitop245,hitop281,hitop141,hitop40,hitop204,hitop21,hitop236,hitop280,hitop84,hitop120,hitop77,hitop92,hitop258,hitop39,hitop254,hitop215,hitop95,hitop106,hitop283,hitop16,hitop20,hitop189,hitop1,hitop136,hitop246,hitop248,hitop257,hitop114,hitop117,hitop250,hitop200,hitop160,hitop23,hitop165,hitop244,hitop9,hitop142,hitop230,hitop149,hitop247,hitop99,hitop66,hitop240,hitop222,hitop90,hitop113,hitop278,hitop203,hitop159,hitop123,hitop275,hitop268,hitop225,hitop143,hitop151,hitop181,hitop211,hitop17,hitop126,hitop5,hitop261,hitop220,hitop15,hitop72,hitop140,hitop109,hitop197,hitop104,hitop_anhedonic_depression,hitop_anxious_worry,hitop_appetite_gain,hitop_appetite_loss,hitop_cognitive_problems,hitop_hyposomnia,hitop_indecisiveness,hitop_insomnia,hitop_panic,hitop_separation_insecurity,hitop_shame_guilt,hitop_situational_phobia,hitop_social_anxiety,hitop_well_being,hitop_sum,whichdata
0,1,1,2,3,2,1,1,1,0,1,2,0,0,0,1,3,3,2,0,1,1,1,0,0,0,0,0,0,0,0,0,1,2,0,0,1,3,1,1,0,1,0,0,1,3,2,0,3,0,3,3,1,0,3,0,0,0,1,0,1,0,0,3,1,0,1,0,0,1,1,0,0,0,0,0,1,0,0,0,1,0,0,0,2,11,5,0,6,0,3,1,0,4,2,0,5,28,39,genpop
1,1,0,3,3,0,2,0,2,0,1,3,2,0,1,2,3,3,3,1,3,1,2,1,0,1,3,1,1,0,1,0,1,3,0,2,3,1,2,0,1,1,3,1,1,0,3,2,3,3,3,3,1,1,3,3,1,2,1,3,1,1,1,0,2,0,1,0,1,1,1,1,3,1,2,1,1,1,2,0,1,1,0,2,10,13,5,2,6,5,3,4,12,3,3,10,17,27,93,genpop
2,1,1,2,2,2,1,1,2,1,1,1,3,1,2,1,1,2,1,2,1,1,2,2,2,1,2,1,2,1,1,2,1,3,3,2,1,1,2,0,1,2,1,1,3,2,1,2,1,3,2,1,2,2,1,1,1,1,2,1,2,2,1,2,1,2,2,1,2,1,1,1,2,2,1,1,1,3,2,1,2,1,0,0,15,12,6,6,5,5,4,5,8,8,7,9,18,16,108,genpop
3,1,0,0,2,1,0,0,0,0,0,0,0,0,0,0,2,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,2,3,3,2,1,0,2,1,0,0,0,0,0,0,0,0,0,0,0,3,0,0,0,0,0,1,0,0,1,0,1,0,0,0,0,0,1,0,3,0,1,0,0,6,1,0,0,4,1,13,17,genpop
4,3,0,1,3,3,3,3,0,1,2,0,1,0,2,2,1,0,3,2,1,1,2,0,1,3,2,1,1,3,3,1,1,0,0,0,1,2,1,0,1,2,3,1,1,0,2,3,1,0,0,1,3,0,0,1,2,0,0,0,1,0,0,2,2,0,3,1,1,1,2,0,1,1,1,0,3,1,1,1,1,0,0,2,16,6,12,0,9,2,3,10,9,8,4,2,11,8,92,genpop


In [47]:
data_genpop_enriched_full.head()

,caseid,weight,sample_type,screener_1ongoing,screener_2impact,screener_3depression,screener_4anxiety,screener_5attention,consent,mood_yn,FNM_Q25_1,FNM_Q25_2,FNM_Q25_3,FNM_Q25_4,FNM_Q25_5,FNM_Q25_6,FNM_Q25_955,FNM_Q25_933,FNM_Q25_open1,mood_years,FNM_Q27_1,FNM_Q27_2,FNM_Q27_3,FNM_Q27_4,FNM_Q27_5,FNM_Q27_6,FNM_Q27_7,FNM_Q27_8,FNM_Q27_9,FNM_Q27_10,FNM_Q27_11,FNM_Q27_12,FNM_Q27_13,FNM_Q27_14,FNM_Q27_15,FNM_Q27_16,FNM_Q27_17,FNM_Q27_18,FNM_Q27_19,FNM_Q27_955,FNM_Q27_933,FNM_Q27_open1,mood_bothered_orig,anxiety_yn,FNM_Q30_m_1,FNM_Q30_m_2,FNM_Q30_m_3,FNM_Q30_m_4,FNM_Q30_m_5,FNM_Q30_m_6,FNM_Q30_m_7,FNM_Q30_m_8,FNM_Q30_m_955,FNM_Q30_m_933,FNM_Q30_open1,anxiety_years,FNM_Q32_1,FNM_Q32_2,FNM_Q32_3,FNM_Q32_4,FNM_Q32_5,FNM_Q32_6,FNM_Q32_7,FNM_Q32_8,FNM_Q32_9,FNM_Q32_10,FNM_Q32_955,FNM_Q32_933,FNM_Q32_open1,anxiety_bothered_orig,attention_yn,FNM_Q35_m_1,FNM_Q35_m_2,FNM_Q35_m_3,FNM_Q35_m_933,FNM_Q35_open1,attention_years,FNM_Q37_m_1,FNM_Q37_m_2,FNM_Q37_m_3,FNM_Q37_m_4,FNM_Q37_m_5,FNM_Q37_m_6,FNM_Q37_m_7,FNM_Q37_m_8,FNM_Q37_m_9,FNM_Q37_m_955,FNM_Q37_m_933,FNM_Q37_open1,attention_bothered_orig,inattention_1,inattention_2,inattention_3,inattention_4,inattention_5,inattention_6,FNM_Q1_attn,inattention_7,inattention_8,inattention_9,hyperactivity_1,hyperactivity_2,hyperactivity_3,hyperactivity_4,hyperactivity_5,impulsivity_1,impulsivity_2,impulsivity_3,impulsivity_4,sct_1,sct_2,sct_3,sct_4,sct_5,sct_6,sct_7,sct_8,sct_9,gad_1,gad_2,gad_3,gad_4,gad_5,gad_6,gad_7,phq_1,phq_2,phq_3,phq_4,phq_5,phq_6,phq_7,phq_8,hitop157,hitop81,hitop34,hitop54,hitop243,hitop182,hitop69,hitop89,hitop50,check_moderately,hitop129,hitop265,hitop124,hitop231,hitop93,hitop67,hitop245,hitop281,hitop141,hitop40,hitop204,hitop21,hitop236,hitop280,hitop84,hitop120,hitop77,hitop92,hitop258,hitop39,hitop254,hitop215,hitop95,hitop106,hitop283,hitop16,hitop20,hitop189,hitop1,hitop136,hitop246,hitop248,hitop257,hitop114,hitop117,hitop250,hitop200,hitop160,hitop23,hitop165,hitop244,hitop9,hitop142,hitop230,hitop149,hitop247,hitop99,hitop66,hitop240,hitop222,hitop90,hitop113,hitop278,hitop203,hitop159,hitop123,hitop275,hitop268,hitop225,hitop143,hitop151,hitop181,hitop211,hitop17,hitop126,hitop5,hitop261,hitop220,check_notatall,hitop15,hitop72,hitop140,hitop109,hitop197,hitop104,todayinattention_1,todayinattention_2,todayinattention_3,todayinattention_4,todayinattention_5,todayinattention_6,todayinattention_7,todayinattention_8,todayinattention_9,todayhyperactivity_1,todayhyperactivity_2,todayhyperactivity_3,todayhyperactivity_4,todayhyperactivity_5,todayimpulsivity_1,todayimpulsivity_2,todayimpulsivity_3,todayimpulsivity_4,todaysct_1,todaysct_2,todaysct_3,todaysct_4,todaysct_5,todaysct_6,todaysct_7,todaysct_8,todaysct_9,today_na1,todaygad_1,todaygad_2,todaygad_3,todaygad_4,...,hitop117_recontact,hitop250_recontact,hitop200_recontact,hitop160_recontact,hitop23_recontact,hitop165_recontact,hitop244_recontact,hitop9_recontact,hitop142_recontact,hitop230_recontact,hitop149_recontact,hitop247_recontact,hitop99_recontact,hitop66_recontact,hitop240_recontact,hitop222_recontact,hitop90_recontact,hitop113_recontact,hitop278_recontact,hitop203_recontact,hitop159_recontact,hitop123_recontact,hitop275_recontact,hitop268_recontact,hitop225_recontact,hitop143_recontact,hitop151_recontact,hitop181_recontact,hitop211_recontact,hitop17_recontact,hitop126_recontact,hitop5_recontact,hitop261_recontact,hitop220_recontact,check_notatall_recontact,hitop15_recontact,hitop72_recontact,hitop140_recontact,hitop109_recontact,hitop197_recontact,hitop104_recontact,todayinattention_1_recontact,todayinattention_2_recontact,todayinattention_3_recontact,todayinattention_4_recontact,todayinattention_5_recontact,todayinattention_6_recontact,todayinattention_7_recontact,todayinattention_8_recontact,todayinattention_9_recontact,todayhyperactivity_1_recontact,todayhyperactivity_2_recontact,todayhyperactivity_3_recontact,todayhyperactivity_4_recontact,todayhyperactivity_5_recontact,todayimpulsivity_1_recontact,todayimpuls

In [53]:
for c in data_genpop_enriched_full.columns:
    print(c)

caseid
weight
sample_type
screener_1ongoing
screener_2impact
screener_3depression
screener_4anxiety
screener_5attention
consent
mood_yn
FNM_Q25_1
FNM_Q25_2
FNM_Q25_3
FNM_Q25_4
FNM_Q25_5
FNM_Q25_6
FNM_Q25_955
FNM_Q25_933
FNM_Q25_open1
mood_years
FNM_Q27_1
FNM_Q27_2
FNM_Q27_3
FNM_Q27_4
FNM_Q27_5
FNM_Q27_6
FNM_Q27_7
FNM_Q27_8
FNM_Q27_9
FNM_Q27_10
FNM_Q27_11
FNM_Q27_12
FNM_Q27_13
FNM_Q27_14
FNM_Q27_15
FNM_Q27_16
FNM_Q27_17
FNM_Q27_18
FNM_Q27_19
FNM_Q27_955
FNM_Q27_933
FNM_Q27_open1
mood_bothered_orig
anxiety_yn
FNM_Q30_m_1
FNM_Q30_m_2
FNM_Q30_m_3
FNM_Q30_m_4
FNM_Q30_m_5
FNM_Q30_m_6
FNM_Q30_m_7
FNM_Q30_m_8
FNM_Q30_m_955
FNM_Q30_m_933
FNM_Q30_open1
anxiety_years
FNM_Q32_1
FNM_Q32_2
FNM_Q32_3
FNM_Q32_4
FNM_Q32_5
FNM_Q32_6
FNM_Q32_7
FNM_Q32_8
FNM_Q32_9
FNM_Q32_10
FNM_Q32_955
FNM_Q32_933
FNM_Q32_open1
anxiety_bothered_orig
attention_yn
FNM_Q35_m_1
FNM_Q35_m_2
FNM_Q35_m_3
FNM_Q35_m_933
FNM_Q35_open1
attention_years
FNM_Q37_m_1
FNM_Q37_m_2
FNM_Q37_m_3
FNM_Q37_m_4
FNM_Q37_m_5
FNM_Q37_m_6
FNM_Q37_m

In [16]:
# TO REMOVE: quickly check loaded data
print('val - genpop')
flag_metric, pconfig, pmetric, pscalar, pstrict = cfa_helper_func(
    scalename = 'insomnia',
    list_of_items = ['hitop160', 'hitop254', 'hitop261', 'hitop268'],
    do_metric = True, do_scalar = True, do_strict = True, 
    mydata_python = mydata_val_enriched, 
    mydata_temp_path = '/Users/zeleninam2/Documents/1_projects/hitop/data/CFA/temp/cfa_temp.csv')

val - genpop


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.




CONFIG chisq p = 0.004
PASSED SECONDARY CRITERIA WITH CFI = 0.998 TLI = 0.995 RMSEA = 0.055


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.




METRIC INVARIANT chisq p = 0.652
SCALAR INVARIANT NOT PASSED, chisq p = 0
STRICT N/A


# Do main CFA analysis

## 3-way CFA


### 1. Check original items
Get full 3-way CFA results (config, metric, scalar, strict) for all original scales.

In [16]:
orig_items = {'anhedonic_depression': 'anhedonic_depression =~hitop39 + hitop77 + hitop84 + hitop92 + hitop93 + hitop123 + hitop157 + hitop182 + hitop230 + hitop246',
         'anxious_worry': 'anxious_worry =~hitop20 + hitop34 + hitop89 + hitop203 + hitop248 + hitop265',
         'appetite_gain': 'appetite_gain =~hitop120 + hitop141 + hitop243 + hitop275',
         'appetite_loss': 'appetite_loss =~hitop280 + hitop283 + hitop109',
         'cognitive_problems': 'cognitive_problems =~hitop67 + hitop189 + hitop142',
         'hyposomnia': 'hyposomnia =~hitop99 + hitop181 + hitop5 + hitop66 + hitop231',
         'insomnia': 'insomnia =~hitop160 + hitop254 + hitop261 + hitop268',
         'panic': 'panic =~hitop15 + hitop104 + hitop126 + hitop211 + hitop215 + hitop257',
         'separation_insecurity': 'separation_insecurity =~hitop40 + hitop50 + hitop69 + hitop81 + hitop113 + hitop136 + hitop151 + hitop197',
         'shame_guilt': 'shame_guilt =~hitop72 + hitop140 + hitop143 + hitop220',
         'situational_phobia': 'situational_phobia =~hitop16 + hitop165 + hitop225 + hitop247 + hitop278',
         'social_anxiety': 'social_anxiety =~hitop1 + hitop17 + hitop114 + hitop117 + hitop124 + hitop129 + hitop204 + hitop222 + hitop236 + hitop258',
         'well_being': 'well_being =~hitop9 + hitop23 + hitop54 + hitop106 + hitop149 + hitop200 + hitop244 + hitop245 + hitop250 + hitop281'}


In [37]:
with open("logs/mylog_3wayCFA_origscales_seed12345_v2.txt", "w") as f:
    with redirect_stdout(f):
        for scale, items in orig_items.items():
            # print which scale we are processing through R - this way it doesn't get saved in the log file
            ro.globalenv['scale_to_print'] = scale
            ro.r('print(scale_to_print)')
            # create a neat list of items to test for this scale
            items_only = items.split("=~",1)[1]
            items_list = items_only.split(" + ")
            # test   
            run_specific_cfa(whichscale=scale,\
                         item_list=items_list, \
                         whichcfa='strict') 

R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be perm

### 2. Find invariant subsets

#### Looping through all scales where full scales did not pass metric invariance:

In [74]:
with open("logs/mylog_3wayCFA_lookingforinv_seed12345_v1.txt", "w") as f:
    with redirect_stdout(f):
        for scale in ['anhedonic_depression', 'anxious_worry', 'appetite_gain', 'separation_insecurity', 'situation_phobia', 'social_anxiety', 'well_being']:
            print(f'\n\n\n======================================\nTESTING SCALE {scale.upper()}\n======================================\n')
            items = orig_items[scale]
            # create a neat FULL list of items to test for this scale
            items_only = items.split("=~",1)[1]
            items_list = items_only.split(" + ")
            # how many items are there in total?
            howmany = len(items_list)
            # remove items one by one
            for i in range (1, howmany):
                how_many_to_try = howmany - i
                print(f'\n-----------------------------\nTESTING n - {i} = {how_many_to_try} ITEMS for scale {scale}\n-----------------------------\n')
                if how_many_to_try <= 2: # the min amount of items we can test is 3
                    print("\nWe ran out of items! No inv subset can be found")
                    break
                # test cfa
                successful_combinations_for_scale = do_three_way_cfa_ablations(whichscale=scale, whichcfa='metric', howmanyitems=how_many_to_try)
                # if found any number of successful items, save them and stop trying for this scale
                if successful_combinations_for_scale: 
                    print(f'\n!!!!! Found at least one invariant subset for SCALE {scale} with ITEMS = {how_many_to_try} (removing {i} items)\n')
                    print(successful_combinations_for_scale)
                    break

R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be perm

#### An option to do it manually:
For each scale, start with x-1 items, then x-2 etc; check all compbinations of those items.

Running only down to metric level because this can get very computationally-heavy. For the selected inv subsets, I'll then test all CFA levels later.

In [75]:
successful_combinations_for_scale = do_three_way_cfa_ablations(whichscale='social_anxiety', whichcfa='metric', howmanyitems=6)
successful_combinations_for_scale

R[write to console]: No AFIs were selected, so only chi-squared will be permuted.




FOR SCALE SOCIAL_ANXIETY:
RUN ALL COMBINATIONS OF 6 ITEMS and checks if a combination is 3-way invariant.


Running SOCIAL_ANXIETY
Items: social_anxiety =~hitop1 + hitop17 + hitop114 + hitop117 + hitop124 + hitop129 + hitop204 + hitop222 + hitop236 + hitop258

+++ TESTING 1th COMBINATION out of 210 possible combinations of 6 items +++
Items to test: ('hitop1', 'hitop17', 'hitop114', 'hitop117', 'hitop124', 'hitop129')

 -----> VALID VS GENPOP <----- 


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.




CONFIG INVARIANT NOT PASSED, chisq p = 0
METRIC N/A
SCALAR N/A
STRICT N/A

 -----> VALID VS ENRICHED <----- 


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.




...

R[write to console]: No AFIs were selected, so only chi-squared will be permuted.




METRIC INVARIANT NOT PASSED, chisq p = 0.008
SCALAR N/A
STRICT N/A

 -----> GENPOP VS ENRICHED <----- 


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.




CONFIG INVARIANT chisq p = 0.214
METRIC INVARIANT NOT PASSED, chisq p = 0.002
SCALAR N/A
STRICT N/A

CFA DONE


Found 1 combination(s) of 6 that is 3-way invariant.
{('hitop1', 'hitop17', 'hitop114', 'hitop129', 'hitop204', 'hitop258'): {'val_gp': ['0.138', '0.067', 'NA', 'NA'], 'val_en': ['0.06', '0.066', 'NA', 'NA'], 'gp_en': ['0.167', '0.255', 'NA', 'NA']}}


{('hitop1',
  'hitop17',
  'hitop114',
  'hitop129',
  'hitop204',
  'hitop258'): {'val_gp': ['0.138', '0.067', 'NA', 'NA'], 'val_en': ['0.06',
   '0.066',
   'NA',
   'NA'], 'gp_en': ['0.167', '0.255', 'NA', 'NA']}}

### 3. (if needed) Check invariant subsets

In [81]:
##  all inv scales
to_test_inv = {'anhedonic_depression': 'anhedonic_depression =~hitop39 + hitop77 + hitop84 + hitop93 + hitop182 + hitop230 + hitop246',
         'anxious_worry': 'anxious_worry =~hitop34 + hitop89 + hitop203 + hitop248',
         'appetite_gain': 'appetite_gain =~hitop120 + hitop243 + hitop275',
         'appetite_loss': 'appetite_loss =~hitop280 + hitop283 + hitop109',
         'cognitive_problems': 'cognitive_problems =~hitop67 + hitop189 + hitop142',
         'hyposomnia': 'hyposomnia =~hitop99 + hitop181 + hitop5 + hitop66 + hitop231',
         'insomnia': 'insomnia =~hitop160 + hitop254 + hitop261 + hitop268',
         'panic': 'panic =~hitop15 + hitop104 + hitop126 + hitop211 + hitop215 + hitop257',
         'separation_insecurity': 'separation_insecurity =~hitop50 + hitop69 + hitop81 + hitop136 + hitop151 + hitop197',
         'shame_guilt': 'shame_guilt =~hitop72 + hitop140 + hitop143 + hitop220',
         'situational_phobia': 'situational_phobia =~hitop16 + hitop165 + hitop225 + hitop247 + hitop278',
         'social_anxiety': 'social_anxiety =~hitop1 + hitop17 + hitop114 + hitop129 + hitop204 + hitop258',
         'well_being': 'well_being =~hitop9 + hitop23 + hitop149 + hitop200 + hitop244 + hitop250 + hitop281'}

with open("logs/mylog_3wayCFA_invsubsets_seed12345.txt", "w") as f:
    with redirect_stdout(f):
        for scale, items in to_test_inv.items():
            # create a neat list of items to test for this scale
            items_only = items.split("=~",1)[1]
            items_list = items_only.split(" + ")
            # test   
            run_specific_cfa(whichscale=scale,\
                         item_list=items_list, \
                         whichcfa='strict')

R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be perm

RRuntimeError: 

In [82]:
##  all inv scales
to_test_inv = {'social_anxiety': 'social_anxiety =~hitop1 + hitop17 + hitop114 + hitop129 + hitop204 + hitop258',
         'well_being': 'well_being =~hitop9 + hitop23 + hitop149 + hitop200 + hitop244 + hitop250 + hitop281'}

with open("logs/mylog_3wayCFA_invsubsets_seed12345.txt", "a") as f:
    with redirect_stdout(f):
        for scale, items in to_test_inv.items():
            # create a neat list of items to test for this scale
            items_only = items.split("=~",1)[1]
            items_list = items_only.split(" + ")
            # test   
            run_specific_cfa(whichscale=scale,\
                         item_list=items_list, \
                         whichcfa='strict')

R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be perm

In [23]:
to_test_inv = {'situational_phobia': 'situational_phobia =~hitop16 + hitop165 + hitop225 + hitop247'}

with open("logs/mylog_3wayCFA_invsubsets_seed12345.txt", "a") as f:
    with redirect_stdout(f):
        for scale, items in to_test_inv.items():
            # create a neat list of items to test for this scale
            items_only = items.split("=~",1)[1]
            items_list = items_only.split(" + ")
            # test   
            run_specific_cfa(whichscale=scale,\
                         item_list=items_list, \
                         whichcfa='strict')

R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.




## 4. Check exclusion items only

In [77]:
to_test_inv = {'anhedonic_depression': 'anhedonic_depression =~hitop92 + hitop123 + hitop157',
                 'social_anxiety': 'social_anxiety =~hitop117 + hitop124 + hitop222 + hitop236',
                 'well_being': 'well_being =~hitop54 + hitop200 + hitop245',
                 'well_being': 'well_being =~hitop54 + hitop106 + hitop245',
                 'well_being': 'well_being =~hitop23 + hitop54 + hitop245'}

with open("logs/mylog_3wayCFA_excludedonly.txt", "w") as f:
    with redirect_stdout(f):
        for scale, items in to_test_inv.items():
            # create a neat list of items to test for this scale
            items_only = items.split("=~",1)[1]
            items_list = items_only.split(" + ")
            # test   
            run_specific_cfa(whichscale=scale,\
                         item_list=items_list, \
                         whichcfa='strict')

R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be perm

In [80]:
to_test_inv = {'well_being1': 'well_being =~hitop54 + hitop200 + hitop245',
                 'well_being2': 'well_being =~hitop54 + hitop106 + hitop245',
                 'well_being3': 'well_being =~hitop23 + hitop54 + hitop245'}

with open("logs/mylog_3wayCFA_excludedonly.txt", "a") as f:
    with redirect_stdout(f):
        for scale, items in to_test_inv.items():
            # create a neat list of items to test for this scale
            items_only = items.split("=~",1)[1]
            items_list = items_only.split(" + ")
            # test   
            run_specific_cfa(whichscale=scale,\
                         item_list=items_list, \
                         whichcfa='strict')

R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be perm

## For interpretation - check what items mean

In [32]:
scales = {
    'items_anhedonic_depression':['39','77','84','92','93','123','157','182','230','246'],
    'items_anxious_worry': ['20','34','89','203','248','265'],
    'items_appetite_gain': ['120','141','243','275'],
    'items_appetite_loss': ['280','283','109'],
    'items_cognitive_problems': ['67','189','142'],
    'items_hyposomnia': ['99','181','5','66','231'],
    'items_insomnia': ['160','254','261','268'],
    'items_panic': ['15','104','126','211','215','257'],
    'items_separation_insecurity': ['40','50','69','81','113','136','151','197'],
    'items_shame_guilt': ['72','140','143','220'],
    'items_situational_phobia': ['16','165','225','247','278'],
    'items_social_anxiety': ['1','17','114','117','124','129','204','222','236','258'],
    'items_well_being': ['9','23','54','106','149','200','244','245','250','281']}

for scale, items in scales.items():
    print(scale)
    check_hitop_ids(items, my_item_lookup=item_lookup)

items_anhedonic_depression
   hitop39   It felt like there wasn’t anything interesting or fun to do.
   hitop77   I didn’t look forward to seeing friends or family.
   hitop84   I felt depressed.
   hitop92   It took a lot of effort to do everyday activities.
   hitop93   Nothing seemed interesting to me.
   hitop123   Nothing made me laugh.
   hitop157   I had very little energy.
   hitop182   I was unable to enjoy things like I normally do.
   hitop230   I felt emotionally numb.
   hitop246   I was a lot less talkative than usual.


items_anxious_worry
   hitop20   I felt tense.
   hitop34   Thoughts were racing through my head.
   hitop89   I had a lot of nervous energy.
   hitop203   I felt very stressed.
   hitop248   I worried about almost everything.
   hitop265   I was overwhelmed by anxiety.


items_appetite_gain
   hitop120   I could not keep myself from eating.
   hitop141   I thought a lot about food.
   hitop243   I stuffed myself with food.
   hitop275   I ate even when I

# Whatever else random stuff I want to do

In [16]:
anh_dep_inv = ['hitop39','hitop77','hitop84', 'hitop93','hitop182','hitop230','hitop246']
run_specific_cfa(whichscale='anhedonic_depression',\
                item_list=anh_dep_inv, \
                whichcfa='strict') 



FOR SCALE ANHEDONIC_DEPRESSION:

RUN ALL 3-way CFA for items: ['hitop39', 'hitop77', 'hitop84', 'hitop93', 'hitop182', 'hitop230', 'hitop246'].

 -----> VALID VS GENPOP <----- 


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.




CONFIG chisq p = 0
PASSED SECONDARY CRITERIA WITH CFI = 0.995 TLI = 0.992 RMSEA = 0.041


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.




METRIC INVARIANT chisq p = 0.05


...

R[write to console]: No AFIs were selected, so only chi-squared will be permuted.




CONFIG INVARIANT chisq p = 0.34


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.




METRIC INVARIANT chisq p = 0.069
SCALAR INVARIANT NOT PASSED, chisq p = 0
STRICT N/A

!!! PASSES 3-WAY INVARIANCE
DONE


0

In [27]:
with open("logs/mylog_3wayCFA_lookingforinv_anhdep_6items_seed12345.txt", "w") as f:
    with redirect_stdout(f):
        for scale in ['anhedonic_depression']:
            print(f'\n\n\n======================================\nTESTING SCALE {scale.upper()}\n======================================\n')
            items = orig_items[scale]
            # create a neat FULL list of items to test for this scale
            items_only = items.split("=~",1)[1]
            items_list = items_only.split(" + ")
            # how many items are there in total?
            howmany = len(items_list)
            how_many_to_try = 6
            print(f'\n-----------------------------\nTESTING n - {i} = {how_many_to_try} ITEMS for scale {scale}\n-----------------------------\n')
            # test cfa
            successful_combinations_for_scale = do_three_way_cfa_ablations(whichscale=scale, whichcfa='metric', howmanyitems=how_many_to_try)
            # if found any number of successful items, save them and stop trying for this scale
            if successful_combinations_for_scale: 
                print(f'\n!!!!! Found at least one invariant subset for SCALE {scale} with ITEMS = {how_many_to_try} (removing {i} items)\n')
                print(successful_combinations_for_scale)
                break

R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be perm

In [22]:
with open("logs/mylog_3wayCFA_lookingforinv_seed12345_v2.txt", "a") as f:
    with redirect_stdout(f):
        for scale in ['situational_phobia']:
            print(f'\n\n\n======================================\nTESTING SCALE {scale.upper()}\n======================================\n')
            items = orig_items[scale]
            # create a neat FULL list of items to test for this scale
            items_only = items.split("=~",1)[1]
            items_list = items_only.split(" + ")
            # how many items are there in total?
            howmany = len(items_list)
            # remove items one by one
            for i in range (1, howmany):
                how_many_to_try = howmany - i
                print(f'\n-----------------------------\nTESTING n - {i} = {how_many_to_try} ITEMS for scale {scale}\n-----------------------------\n')
                if how_many_to_try <= 2: # the min amount of items we can test is 3
                    print("\nWe ran out of items! No inv subset can be found")
                    break
                # test cfa
                successful_combinations_for_scale = do_three_way_cfa_ablations(whichscale=scale, whichcfa='metric', howmanyitems=how_many_to_try)
                # if found any number of successful items, save them and stop trying for this scale
                if successful_combinations_for_scale: 
                    print(f'\n!!!!! Found at least one invariant subset for SCALE {scale} with ITEMS = {how_many_to_try} (removing {i} items)\n')
                    print(successful_combinations_for_scale)
                    break

R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be perm

In [33]:
run_specific_cfa(whichscale='well_being', item_list=['hitop9','hitop106', 'hitop149', 'hitop200', 'hitop244', 'hitop250', 'hitop281'], whichcfa='strict')

R[write to console]: No AFIs were selected, so only chi-squared will be permuted.






FOR SCALE WELL_BEING:

RUN ALL 3-way CFA for items: ['hitop9', 'hitop106', 'hitop149', 'hitop200', 'hitop244', 'hitop250', 'hitop281'].

 -----> VALID VS GENPOP <----- 


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.




CONFIG chisq p = 0.021
PASSED SECONDARY CRITERIA WITH CFI = 1.0 TLI = 1.001 RMSEA = 0.035


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.




...

R[write to console]: No AFIs were selected, so only chi-squared will be permuted.




CONFIG INVARIANT chisq p = 0.182


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.




METRIC INVARIANT chisq p = 0.05
SCALAR INVARIANT NOT PASSED, chisq p = 0
STRICT N/A

!!! PASSES 3-WAY INVARIANCE
DONE


0

# Analysis for BAARS, GAD, PHQ

In [39]:
for c in data_genpop_enriched_full.columns:
    print(c)

caseid
weight
sample_type
screener_1ongoing
screener_2impact
screener_3depression
screener_4anxiety
screener_5attention
consent
mood_yn
FNM_Q25_1
FNM_Q25_2
FNM_Q25_3
FNM_Q25_4
FNM_Q25_5
FNM_Q25_6
FNM_Q25_955
FNM_Q25_933
FNM_Q25_open1
mood_years
FNM_Q27_1
FNM_Q27_2
FNM_Q27_3
FNM_Q27_4
FNM_Q27_5
FNM_Q27_6
FNM_Q27_7
FNM_Q27_8
FNM_Q27_9
FNM_Q27_10
FNM_Q27_11
FNM_Q27_12
FNM_Q27_13
FNM_Q27_14
FNM_Q27_15
FNM_Q27_16
FNM_Q27_17
FNM_Q27_18
FNM_Q27_19
FNM_Q27_955
FNM_Q27_933
FNM_Q27_open1
mood_bothered
anxiety_yn
FNM_Q30_m_1
FNM_Q30_m_2
FNM_Q30_m_3
FNM_Q30_m_4
FNM_Q30_m_5
FNM_Q30_m_6
FNM_Q30_m_7
FNM_Q30_m_8
FNM_Q30_m_955
FNM_Q30_m_933
FNM_Q30_open1
anxiety_years
FNM_Q32_1
FNM_Q32_2
FNM_Q32_3
FNM_Q32_4
FNM_Q32_5
FNM_Q32_6
FNM_Q32_7
FNM_Q32_8
FNM_Q32_9
FNM_Q32_10
FNM_Q32_955
FNM_Q32_933
FNM_Q32_open1
anxiety_bothered
attention_yn
FNM_Q35_m_1
FNM_Q35_m_2
FNM_Q35_m_3
FNM_Q35_m_933
FNM_Q35_open1
attention_years
FNM_Q37_m_1
FNM_Q37_m_2
FNM_Q37_m_3
FNM_Q37_m_4
FNM_Q37_m_5
FNM_Q37_m_6
FNM_Q37_m_7
FNM_Q37

In [51]:
other_scales = {'hitop_SUM': 'hitop_SUM =~hitop_anhedonic_depression + hitop_anxious_worry + hitop_appetite_gain + hitop_appetite_loss + hitop_cognitive_problems + hitop_hyposomnia + hitop_indecisiveness + hitop_insomnia + hitop_panic + hitop_separation_insecurity + hitop_shame_guilt + hitop_situational_phobia + hitop_social_anxiety', 
    'phq_SUM': 'phq_SUM =~phq_1 + phq_2 + phq_3 + phq_4 + phq_5 + phq_6 + phq_7 + phq_8',
    'gad_SUM': 'gad_SUM =~gad_1 + gad_2 + gad_3 + gad_4 + gad_5 + gad_6 + gad_7',
    'baars_inattention_SUM': 'baars_inattention_SUM =~inattention_1 + inattention_2 + inattention_3 + inattention_4 + inattention_5 + inattention_6 + inattention_7 + inattention_8 + inattention_9',
    'baars_hyperactivity_SUM': 'baars_hyperactivity_SUM =~hyperactivity_1 + hyperactivity_2 + hyperactivity_3 + hyperactivity_4 + hyperactivity_5',
    'baars_impulsivity_SUM': 'baars_impulsivity_SUM =~impulsivity_1 + impulsivity_2 + impulsivity_3 + impulsivity_4',
    'baars_sct_SUM': 'baars_sct_SUM =~sct_1 + sct_2 + sct_3 + sct_4 + sct_5 + sct_6 + sct_7 + sct_8 + sct_9'} 

with open("logs/mylog_3wayCFA_baarsgadphq_seed12345.txt", "w") as f:
    with redirect_stdout(f):
        for scale, items in other_scales.items():
            print(f'\n\n\n======================================\nTESTING SCALE {scale.upper()}\n======================================\n')
            print(f"Items: {items}")
        
            items_only = items.split("=~",1)[1]
            items_list = items_only.split(" + ")
        
            # +++ GENPOP VS ENRICHED +++ 
            print('\n -----> GENPOP VS ENRICHED <----- ')
            flag_metric_gp_en, pconfig_gp_en, pmetric_gp_en, pscalar_gp_en, pstrict_gp_en = cfa_helper_func(\
                            scalename = scale,\
                            list_of_items = items_list,\
                            do_metric = True, \
                            do_scalar = True, \
                            do_strict = True, \
                            mydata_python = data_genpop_enriched_full, \
                            mydata_temp_path = path_to_helpfile)    

R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be perm